# Specification curve figures

Reads a completed results directory and writes the two figures used in the README.
This notebook produces figures only. It contains no analysis and no modelling;
every number it draws is computed by `pisa_specsens` and read from disk.

Run the grid first:

    pisa-specsens --data data/uk_pisa_2022.csv --out results/v2

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("../results/v2")
FIGURES = Path("../figures")
FIGURES.mkdir(exist_ok=True)

grid = pd.read_csv(RESULTS / "grid_results.csv")
summary = json.loads((RESULTS / "summary.json").read_text())
len(grid)

## Specification curve: performance across the grid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, form in zip(axes, ["binary", "continuous"]):
    subset = grid[grid.target_form == form].sort_values("estimate").reset_index(drop=True)
    ax.errorbar(
        range(len(subset)), subset.estimate,
        yerr=[subset.estimate - subset.ci_low, subset.ci_high - subset.estimate],
        fmt="o", markersize=4, linewidth=1, capsize=2, color="#333333",
    )
    ax.set_title(f"{form} target ({subset.metric_name.iloc[0]})")
    ax.set_xlabel("specification, ordered by estimate")
    ax.set_ylabel(subset.metric_name.iloc[0])
    ax.grid(alpha=0.3, linewidth=0.5)

fig.tight_layout()
fig.savefig(FIGURES / "specification_curve.png", dpi=200)

## Rank of ESCS across the grid

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = grid.focal_rank.value_counts().sort_index()
ax.bar(counts.index, counts.values, color="#333333")
ax.set_xlabel("rank of ESCS by permutation importance")
ax.set_ylabel("number of specifications")
ax.set_title("Where ESCS places, across 48 defensible specifications")
ax.grid(alpha=0.3, linewidth=0.5, axis="y")
fig.tight_layout()
fig.savefig(FIGURES / "escs_rank.png", dpi=200)

## Blocks permuted jointly against single features

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))

order = grid.sort_values("block_disposition").reset_index(drop=True)
ax.plot(range(len(order)), order.block_disposition, "o", markersize=4,
        color="#1b1b1b", label="disposition block")
ax.plot(range(len(order)), order.block_background, "s", markersize=4,
        color="#9a9a9a", label="background block")
ax.axhline(0, color="#cccccc", linewidth=0.8)
ax.set_xlabel("specification, ordered by disposition block importance")
ax.set_ylabel("drop in score when block is permuted")
ax.set_title("Jointly permuted blocks are stable where single features are not")
ax.legend(frameon=False)
ax.grid(alpha=0.3, linewidth=0.5)
fig.tight_layout()
fig.savefig(FIGURES / "block_importance.png", dpi=200)